# Chunking inspection

This notebook runs the repository's current PDF extraction and chunking implementation on the first four physical pages of the sample PDF, including the reranking list. It only reads the fixture: it does not connect to the database or write application files.

Change `PAGE_LIMIT` below when you want to inspect more pages. After editing repository code, restart the kernel and run all cells to load the new implementation.

In [1]:
from __future__ import annotations

import sys
from html import escape
from pathlib import Path
from uuid import NAMESPACE_URL, uuid5

from IPython.display import HTML, display

sys.dont_write_bytecode = True


def find_repository_root() -> Path:
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / "pyproject.toml").is_file() and (
            candidate / "src" / "rag_assistant"
        ).is_dir():
            return candidate
    raise RuntimeError("Run this notebook from somewhere inside the repository.")


REPOSITORY_ROOT = find_repository_root()
SOURCE_DIRECTORY = REPOSITORY_ROOT / "src"
if str(SOURCE_DIRECTORY) not in sys.path:
    sys.path.insert(0, str(SOURCE_DIRECTORY))

In [2]:
from rag_assistant.core.config import Settings
from rag_assistant.extraction import PdfExtractor
from rag_assistant.indexing.chunking import (
    CHUNKING_VERSION,
    ChunkingParagraph,
    ChunkReference,
    ParagraphChunker,
    TiktokenTokenCounter,
)

PAGE_LIMIT = 4
if not isinstance(PAGE_LIMIT, int) or PAGE_LIMIT < 1:
    raise ValueError("PAGE_LIMIT must be a positive integer.")
PDF_PATH = REPOSITORY_ROOT / "tests" / "fixtures" / "documents" / "sample_doc.pdf"
EMBEDDING_MODEL = str(Settings.model_fields["embedding_model"].default)

print(f"PDF: {PDF_PATH.name}")
print(f"Physical pages selected: 1-{PAGE_LIMIT}")
print(f"Tokenizer model: {EMBEDDING_MODEL}")

PDF: sample_doc.pdf
Physical pages selected: 1-4
Tokenizer model: openai/text-embedding-3-large


In [3]:
extracted = PdfExtractor().extract(PDF_PATH.read_bytes())
selected_source_blocks = [
    block for block in extracted.paragraphs if block.page_number <= PAGE_LIMIT
]

# The production pipeline normally supplies persisted UUIDs. Deterministic temporary IDs
# are sufficient here because this notebook never persists the chunks.
paragraphs = [
    ChunkingParagraph(
        source_paragraph_id=uuid5(
            NAMESPACE_URL,
            f"{PDF_PATH.name}:page:{block.page_number}:paragraph:{block.paragraph_number}",
        ),
        page_number=block.page_number,
        paragraph_number=block.paragraph_number,
        kind=block.kind,
        normalized_text=block.normalized_text,
    )
    for block in selected_source_blocks
]

print(f"Extracted {len(paragraphs)} source paragraphs from {PAGE_LIMIT} pages.")

Extracted 29 source paragraphs from 4 pages.


In [4]:
# No chunking rules are reproduced here: this calls the current repository implementation
# with its current default target, maximum, and overlap settings.
chunker = ParagraphChunker(TiktokenTokenCounter(EMBEDDING_MODEL))
chunks = chunker.chunk(paragraphs)

settings = chunks[0].metadata
print(f"Created {len(chunks)} chunks.")
print(f"Method: {CHUNKING_VERSION}")
print(
    "Settings: "
    f"target={settings['target_tokens']} tokens, "
    f"maximum={settings['max_tokens']} tokens, "
    f"paragraph overlap={settings['paragraph_overlap']}"
)

Created 10 chunks.
Method: structure-aware-v2
Settings: target=400 tokens, maximum=600 tokens, paragraph overlap=1


## Resulting chunks

Each card shows the complete normalized text sent for embedding and the persisted citation units it represents. Offsets appear only when an oversized source paragraph was split.

In [5]:
def reference_label(reference: ChunkReference) -> str:
    label = f"page {reference.page_number}, paragraph {reference.paragraph_number}"
    if reference.start_offset is not None:
        label += f" [{reference.start_offset}:{reference.end_offset})"
    return label


card_style = "border:1px solid #d0d7de;border-radius:8px;padding:12px 16px;margin:12px 0"
text_style = (
    "text-align:right;line-height:1.9;background:#f6f8fa;color:#1f2328;"
    "padding:12px;border-radius:6px"
)

for chunk in chunks:
    sources = "; ".join(reference_label(reference) for reference in chunk.references)
    target = int(chunk.metadata["target_tokens"])
    size_note = "above target, within hard limit" if chunk.token_count > target else "within target"
    rendered_text = escape(chunk.normalized_text).replace("\n", "<br>")
    display(
        HTML(
            f"""
            <section style="{card_style}">
              <h3 style="margin:0 0 6px">Chunk {chunk.ordinal}</h3>
              <div><strong>{chunk.token_count} tokens</strong> — {size_note}</div>
              <div style="margin:4px 0 12px"><strong>Sources:</strong> {escape(sources)}</div>
              <div dir="rtl" style="{text_style}">
                {rendered_text}
              </div>
            </section>
            """
        )
    )